In [ ]:
from utils_sm import *
from utils_SDEMEM import *
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import math
import random
import matplotlib.pyplot as plt
import torch.optim as optim
from tqdm import tqdm
import time

In [ ]:
var_name_aug_latex = [
    r"$\log m_0$",
    r"$\log \mathrm{scale}$",
    r"$\log \mathrm{offset}$",
    r"$\log \sigma$",
    r"$\mu_{\delta}$",
    r"$\mu_{\gamma}$",
    r"$\mu_{k}$",
    r"$\mu_{t_0}$",
    r"$\log \tau_{\delta}$",
    r"$\log \tau_{\gamma}$",
    r"$\log \tau_{k}$",
    r"$\log \tau_{t_0}$",
    r"$\log \mathrm{scale} + \log m_0 + \mu_k$",
]

In [ ]:
# ===== Setting ===== #
obs_size = 200
T = 30
theta_dim = 12
x_dim = 180

prior_mean = torch.tensor([5, 1, 3, -1.5, -0.694, -3, 0.027, 0, -0.8, -0.8, -0.8, -0.8], dtype = torch.float32)
prior_std = torch.tensor([1, 1, 1, 1, 0.6, 0.5, 1, 1, 0.5, 0.5, 0.5, 0.5], dtype = torch.float32)

# [log_m0, log_scale, log_offset, log_sigma, mu_delta, mu_gamma, mu_k, mu_t0, log_tau_delta, log_tau_gamma, log_tau_k, log_tau_t0]
theta_true = torch.tensor([5.7, 0.7, 2.08, -1.6, -0.694, -3, 0.027, 0, -1.15, -1.15, -1.15, -1.15], dtype = torch.float32).reshape(1, -1)


var_name = ['log_m0', 'log_scale', 'log_offset', 'log_sigma', 'mu_delta', 'mu_gamma', 'mu_k', 'mu_t0', 'log_tau_delta', 'log_tau_gamma', 'log_tau_k', 'log_tau_t0']
var_name_aug = ['log_m0', 'log_scale', 'log_offset', 'log_sigma', 'mu_delta', 'mu_gamma', 'mu_k', 'mu_t0', 'log_tau_delta', 'log_tau_gamma', 'log_tau_k', 'log_tau_t0', 'log_m0 + mu_k']

In [ ]:
theta_true_aug = torch.cat([theta_true.ravel(), (theta_true[0, 0] + theta_true[0, 1] + theta_true[0, 6]).unsqueeze(0)]).ravel()

## Table

In [ ]:
def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

def weighted_quantile(array, weight, quantile):
    # weight sums to 1
    sorter = np.argsort(array)
    sorted_array = array[sorter]
    sorted_weight = weight[sorter]
    index = np.searchsorted(np.cumsum(sorted_weight), quantile, side='left')
    return sorted_array[index]

In [ ]:
# ====== Our method ====== #
error_ours = torch.zeros(100, theta_dim + 1)
width_ours = torch.zeros(100, theta_dim + 1)
cover_ours = torch.zeros(100, theta_dim + 1)

for obs_id in range(100):
    samples = torch.from_numpy( np.load(f"res_inference/sm_round1/samples{obs_id}.npy"))
    theta_post = samples[:, 10000::100, :].reshape(-1, theta_dim)

    theta_post_aug = torch.cat([theta_post, (theta_post[:, 0] + theta_post[:, 1] + theta_post[:, 6]).unsqueeze(1)], dim = 1)

    error_ours[obs_id] = (theta_post_aug.mean(dim = 0) - theta_true_aug).abs()
    CI_lower = torch.quantile(theta_post_aug, 0.025, dim = 0)
    CI_upper = torch.quantile(theta_post_aug, 0.975, dim = 0)
    width_ours[obs_id] = CI_upper - CI_lower
    cover_ours[obs_id] = ((theta_true_aug >= CI_lower) & (theta_true_aug <= CI_upper)).float()

In [ ]:
error_ours.mean(dim = 0), width_ours.mean(dim = 0), cover_ours.mean(dim = 0)

In [ ]:
from torch.distributions import MultivariateNormal

# ====== ABC ====== #
error_ABC = torch.zeros(100, theta_dim + 1)
width_ABC = torch.zeros(100, theta_dim + 1)
cover_ABC = torch.zeros(100, theta_dim + 1)
for obs_id in range(100):
    # To construct the proposal and prior distributions in order to get the weight
    # Load SW data
    task_id = obs_id % 10
    theta_SW1 = np.load(f"res_SW1/theta_SW1_task{task_id}.npy")[:100]
    theta_SW1 = torch.tensor(theta_SW1, dtype=torch.float32)

    prop_mean = theta_SW1.mean(dim = 0, keepdims = True)
    prop_std = theta_SW1.std(dim = 0, keepdims = True) 
    prop_std *= 2
    prop_std = prop_std.clamp_min(1e-8)

    dist_prop = MultivariateNormal(loc = prop_mean.ravel(), covariance_matrix = torch.diag(prop_std.ravel()**2))
    dist_prior = MultivariateNormal(loc = prior_mean.ravel(), covariance_matrix = torch.diag(prior_std.ravel()**2))

    # Load ABC results
    theta_set_all = [torch.from_numpy(np.load(f"res_ABC/theta_set{obs_id}_batch{batch_id}.npy")) for batch_id in range(10)]
    W1_set_all = [torch.from_numpy(np.load(f"res_ABC/W1_set{obs_id}_batch{batch_id}.npy")) for batch_id in range(10)]
    theta_set = torch.cat(theta_set_all)
    W1_set = torch.cat(W1_set_all)
    theta_set = theta_set[W1_set != 0.0] # 0.0 means nan in the simulated data
    W1_set = W1_set[W1_set != 0.0] 

    smallest_values, smallest_indices = torch.topk(W1_set, 10000, largest=False) # top 1000 before
    theta_ABC = theta_set[smallest_indices].clone()
    theta_ABC_aug = torch.cat([theta_ABC, (theta_ABC[:, 0] + theta_ABC[:, 1] + theta_ABC[:, 6]).unsqueeze(1)], dim = 1)

    # Compute the weight
    log_weight_ABC = dist_prior.log_prob(theta_ABC) - dist_prop.log_prob(theta_ABC)
    weight_ABC = normalized_weight(log_weight_ABC)

    # Compute the 3 metrics
    error_ABC[obs_id] = ( (theta_ABC_aug * weight_ABC.unsqueeze(1)).sum(dim = 0) - theta_true_aug.ravel()).abs()

    CI_lower = torch.zeros(theta_dim + 1)
    CI_upper = torch.zeros(theta_dim + 1)
    for j in range(theta_dim + 1):
        CI_lower[j] = float(weighted_quantile(theta_ABC_aug[:, j].numpy(), weight_ABC.numpy(), 0.025))
        CI_upper[j] = float(weighted_quantile(theta_ABC_aug[:, j].numpy(), weight_ABC.numpy(), 0.975))

    width_ABC[obs_id] = (CI_upper - CI_lower).abs()
    cover_ABC[obs_id] = ((theta_true_aug.ravel() >= CI_lower) & (theta_true_aug.ravel() <= CI_upper)).float()    

In [ ]:
error_ABC.mean(dim = 0), width_ABC.mean(dim = 0), cover_ABC.mean(dim = 0)

In [ ]:
# ====== NLE ====== #
cover_NLE = torch.zeros(100, theta_dim+1)
error_NLE = torch.zeros(100, theta_dim + 1)
width_NLE = torch.zeros(100, theta_dim + 1)
for obs_id in range(100):
    theta_post = torch.from_numpy(np.load(f'res_NLE/theta_post_task{obs_id}.npy'))
    if obs_id == 6: # 2 chains failed
        theta_post = theta_post.reshape(4, -1, theta_dim)[[0, 3]].reshape(-1, theta_dim)

    theta_post_aug = torch.cat([theta_post, (theta_post[:, 0]+ theta_post[:, 1] + theta_post[:, 6]).unsqueeze(1)], dim=1)

    error_NLE[obs_id] = (theta_post_aug.mean(dim = 0) - theta_true_aug).abs()

    CI_lower = torch.quantile(theta_post_aug, 0.025, dim=0)
    CI_upper = torch.quantile(theta_post_aug, 0.975, dim=0)

    width_NLE[obs_id] = CI_upper - CI_lower
    cover_NLE[obs_id] = ((theta_true_aug >= CI_lower) & (theta_true_aug <= CI_upper)).float()

In [ ]:
error_NLE.mean(dim = 0), width_NLE.mean(dim = 0), cover_NLE.mean(dim = 0)

In [ ]:
from torch.distributions import MultivariateNormal

# ====== NPE ====== #
cover_NPE = torch.zeros(100, theta_dim + 1)
error_NPE = torch.zeros(100, theta_dim + 1)
width_NPE = torch.zeros(100, theta_dim + 1)
for obs_id in range(100):
    task_id = obs_id % 10

    theta_SW1 = np.load(f"res_SW1/theta_SW1_task{task_id}.npy")[:100]
    theta_SW1 = torch.tensor(theta_SW1, dtype=torch.float32)

    prop_mean = theta_SW1.mean(dim = 0, keepdims = True)
    prop_std = theta_SW1.std(dim = 0, keepdims = True) 
    prop_std *= 2
    prop_std = prop_std.clamp_min(1e-8)

    dist_prop = MultivariateNormal(loc = prop_mean.ravel(), covariance_matrix = torch.diag(prop_std.ravel()**2))
    dist_prior = MultivariateNormal(loc = prior_mean.ravel(), covariance_matrix = torch.diag(prior_std.ravel()**2))

    # load NPE results
    theta_post = torch.from_numpy(np.load(f'res_NPE_newdeepsets/samples_task{obs_id}.npy'))
    theta_post_aug = torch.cat([theta_post, (theta_post[:, 0]+ theta_post[:, 1] + theta_post[:, 6]).unsqueeze(1)], dim=1)

    # Compute the weights for NPE
    log_weight_NPE = dist_prior.log_prob(theta_post) - dist_prop.log_prob(theta_post)
    weight_NPE = normalized_weight(log_weight_NPE)

    # Compute the 3 metrics
    error_NPE[obs_id] = ( (theta_post_aug * weight_NPE.unsqueeze(1)).sum(dim = 0) - theta_true_aug.ravel()).abs()

    CI_lower = torch.zeros(theta_dim + 1)
    CI_upper = torch.zeros(theta_dim + 1)
    for j in range(theta_dim + 1):
        CI_lower[j] = float(weighted_quantile(theta_post_aug[:, j].numpy(), weight_NPE.numpy(), 0.025))
        CI_upper[j] = float(weighted_quantile(theta_post_aug[:, j].numpy(), weight_NPE.numpy(), 0.975))

    width_NPE[obs_id] = (CI_upper - CI_lower).abs()
    cover_NPE[obs_id] = ((theta_true_aug.ravel() >= CI_lower) & (theta_true_aug.ravel() <= CI_upper)).float() 



In [ ]:
error_NPE.mean(dim = 0), width_NPE.mean(dim = 0), cover_NPE.mean(dim = 0)

In [ ]:
methods = ["single-model", "ABC", "NLE", "NPE"]

# Means
error_mean_dict = {
    "single-model": error_ours.mean(dim=0).cpu().numpy(),
    "ABC": error_ABC.mean(dim=0).cpu().numpy(),
    "NLE": error_NLE.mean(dim=0).cpu().numpy(),
    "NPE": error_NPE.mean(dim=0).cpu().numpy(),
}

width_mean_dict = {
    "single-model": width_ours.mean(dim=0).cpu().numpy(),
    "ABC": width_ABC.mean(dim=0).cpu().numpy(),
    "NLE": width_NLE.mean(dim=0).cpu().numpy(),
    "NPE": width_NPE.mean(dim=0).cpu().numpy(),
}

cover_mean_dict = {
    "single-model": cover_ours.mean(dim=0).cpu().numpy(),
    "ABC": cover_ABC.mean(dim=0).cpu().numpy(),
    "NLE": cover_NLE.mean(dim=0).cpu().numpy(),
    "NPE": cover_NPE.mean(dim=0).cpu().numpy(),
}

# Standard deviations across runs
error_std_dict = {
    "single-model": error_ours.std(dim=0).cpu().numpy(),
    "ABC": error_ABC.std(dim=0).cpu().numpy(),
    "NLE": error_NLE.std(dim=0).cpu().numpy(),
    "NPE": error_NPE.std(dim=0).cpu().numpy(),
}

width_std_dict = {
    "single-model": width_ours.std(dim=0).cpu().numpy(),
    "ABC": width_ABC.std(dim=0).cpu().numpy(),
    "NLE": width_NLE.std(dim=0).cpu().numpy(),
    "NPE": width_NPE.std(dim=0).cpu().numpy(),
}

# Keep the same 10 rows as in the current final table
keep_idx = [i for i in range(len(var_name_aug_latex)) if i not in [0, 1, 6]]

# Row labels after dropping rows 0, 1, and 6
row_labels = [
    r"$\log \mathrm{offset}$",
    r"$\log \sigma$",
    r"$\mu_{\delta}$",
    r"$\mu_{\gamma}$",
    r"$\mu_{t_0}$",
    r"$\log \tau_{\delta}$",
    r"$\log \tau_{\gamma}$",
    r"$\log \tau_{k}$",
    r"$\log \tau_{t_0}$",
    r"\begin{tabular}{@{}c@{}} $\log m_0 + \mu_k$ \\ $+\,\log \mathrm{scale}$ \end{tabular}",
]

def two_line_cell(mean, std, bold_mean=False):
    mean_str = rf"\textbf{{{mean:.3f}}}" if bold_mean else rf"{mean:.3f}"
    return rf"\begin{{tabular}}{{@{{}}c@{{}}}} {mean_str} \\ {{\footnotesize ({std:.3f})}} \end{{tabular}}"

def one_line_cell(value):
    return rf"\begin{{tabular}}{{@{{}}c@{{}}}} {value:.2f} \end{{tabular}}"

def one_line_label(label):
    return rf"\begin{{tabular}}{{@{{}}c@{{}}}} {label} \end{{tabular}}"

formatted_labels = []
for i, label in enumerate(row_labels):
    if i == len(row_labels) - 1:
        formatted_labels.append(label)
    else:
        formatted_labels.append(one_line_label(label))

lines = []
lines.append(r"\begin{tabular}{c|cccc|cccc|cccc}")
lines.append(r"\toprule")
lines.append(r" & \multicolumn{4}{c|}{error} & \multicolumn{4}{c|}{width} & \multicolumn{4}{c}{coverage} \\")
lines.append(r" & single-model & ABC & NLE & NPE & single-model & ABC & NLE & NPE & single-model & ABC & NLE & NPE \\")
lines.append(r"\midrule")

for row_pos, idx in enumerate(keep_idx):
    row_cells = [formatted_labels[row_pos]]

    # Bold the minimum mean within the error block
    error_values = [error_mean_dict[method][idx] for method in methods]
    min_error = min(error_values)

    # Bold the minimum width only among methods with coverage > 0.9
    eligible_width_methods = [
        method for method in methods
        if cover_mean_dict[method][idx] > 0.9
    ]

    if eligible_width_methods:
        min_width_eligible = min(width_mean_dict[method][idx] for method in eligible_width_methods)
    else:
        min_width_eligible = None

    for method in methods:
        mean_val = error_mean_dict[method][idx]
        std_val = error_std_dict[method][idx]
        bold_flag = np.isclose(mean_val, min_error)
        row_cells.append(two_line_cell(mean_val, std_val, bold_mean=bold_flag))

    for method in methods:
        mean_val = width_mean_dict[method][idx]
        std_val = width_std_dict[method][idx]

        bold_flag = (
            min_width_eligible is not None
            and cover_mean_dict[method][idx] > 0.9
            and np.isclose(mean_val, min_width_eligible)
        )

        row_cells.append(two_line_cell(mean_val, std_val, bold_mean=bold_flag))

    for method in methods:
        row_cells.append(one_line_cell(cover_mean_dict[method][idx]))

    lines.append(" & ".join(row_cells) + r" \\")
    if row_pos < len(keep_idx) - 1:
        lines.append(r"\midrule")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")

latex_code = "\n".join(lines)
print(latex_code)

## Plot

In [ ]:
task_id = 0
# ours
theta_ours = torch.from_numpy( np.load(f"res_inference/sm_round1/samples{task_id}.npy"))
theta_ours = theta_ours[:, 10000::100, :].reshape(-1, theta_dim)
theta_ours_aug = torch.cat([theta_ours, (theta_ours[:, 0] + theta_ours[:, 1] + theta_ours[:, 6]).unsqueeze(1)], dim = 1)

# ABC
theta_set_all = [torch.from_numpy(np.load(f"res_ABC/theta_set{task_id}_batch{batch_id}.npy")) for batch_id in range(10)]
W1_set_all = [torch.from_numpy(np.load(f"res_ABC/W1_set{task_id}_batch{batch_id}.npy")) for batch_id in range(10)]
theta_set = torch.cat(theta_set_all)
W1_set = torch.cat(W1_set_all)
theta_set = theta_set[W1_set != 0.0] # 0.0 means nan in the simulated data
W1_set = W1_set[W1_set != 0.0] 

smallest_values, smallest_indices = torch.topk(W1_set, 10000, largest=False) # top 1000 before
theta_ABC = theta_set[smallest_indices].clone()
theta_ABC_aug = torch.cat([theta_ABC, (theta_ABC[:, 0] + theta_ABC[:, 1] + theta_ABC[:, 6]).unsqueeze(1)], dim = 1)

# NLE
theta_NLE = torch.from_numpy(np.load(f'res_NLE/theta_post_task{task_id}.npy'))
theta_NLE_aug = torch.cat([theta_NLE, (theta_NLE[:, 0]+ theta_NLE[:, 1] + theta_NLE[:, 6]).unsqueeze(1)], dim=1)

# NPE
theta_NPE = torch.from_numpy(np.load(f'res_NPE_newdeepsets/samples_task{task_id}.npy'))
theta_NPE_aug = torch.cat([theta_NPE, (theta_NPE[:, 0]+ theta_NPE[:, 1] + theta_NPE[:, 6]).unsqueeze(1)], dim=1)

In [ ]:
from torch.distributions import MultivariateNormal
# Calculate weights for ABC and NPE
# To construct the proposal and prior distributions in order to get the weight
# Load SW data

theta_SW1 = np.load(f"res_SW1/theta_SW1_task{task_id}.npy")[:100]
theta_SW1 = torch.tensor(theta_SW1, dtype=torch.float32)

prop_mean = theta_SW1.mean(dim = 0, keepdims = True)
prop_std = theta_SW1.std(dim = 0, keepdims = True) 
prop_std *= 2
prop_std = prop_std.clamp_min(1e-8)

dist_prop = MultivariateNormal(loc = prop_mean.ravel(), covariance_matrix = torch.diag(prop_std.ravel()**2))
dist_prior = MultivariateNormal(loc = prior_mean.ravel(), covariance_matrix = torch.diag(prior_std.ravel()**2))

# Compute the weight
log_weight_ABC = dist_prior.log_prob(theta_ABC) - dist_prop.log_prob(theta_ABC)
weight_ABC = normalized_weight(log_weight_ABC)

log_weight_NPE = dist_prior.log_prob(theta_NPE) - dist_prop.log_prob(theta_NPE)
weight_NPE = normalized_weight(log_weight_NPE)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert to numpy
theta_ours_np = theta_ours_aug.detach().cpu().numpy()
theta_ABC_np  = theta_ABC_aug.detach().cpu().numpy()
theta_NLE_np  = theta_NLE_aug.detach().cpu().numpy()
theta_NPE_np  = theta_NPE_aug.detach().cpu().numpy()

methods = ["single-model", "ABC", "NLE", "NPE"]
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
data_list = [theta_ours_np, theta_ABC_np, theta_NLE_np, theta_NPE_np]

bw_list = [1.0, 1.0, 1.0, 1.0]

var_name_aug_latex = [
    r"$\log m_0$",
    r"$\log \mathrm{scale}$",
    r"$\log \mathrm{offset}$",
    r"$\log \sigma$",
    r"$\mu_{\delta}$",
    r"$\mu_{\gamma}$",
    r"$\mu_{k}$",
    r"$\mu_{t_0}$",
    r"$\log \tau_{\delta}$",
    r"$\log \tau_{\gamma}$",
    r"$\log \tau_{k}$",
    r"$\log \tau_{t_0}$",
    r"$\log \mathrm{scale} + \log m_0 + \mu_k$",
]

plot_dims = [d for d in range(theta_dim + 1) if d not in [0, 1, 6]]

q_low, q_high = 0.001, 0.999
x_margin_ratio = 0.1

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.ravel()

for plot_i, d in enumerate(plot_dims):
    ax = axes[plot_i]

    pooled_d = np.concatenate([data[:, d] for data in data_list])
    pooled_d = pooled_d[np.isfinite(pooled_d)]

    x_low, x_high = np.quantile(pooled_d, [q_low, q_high])

    true_val = float(theta_true_aug[d])
    x_low = min(x_low, true_val)
    x_high = max(x_high, true_val)

    x_margin = x_margin_ratio * (x_high - x_low)
    x_low -= x_margin
    x_high += x_margin

    for j, data in enumerate(data_list):
        sns.kdeplot(
            data[:, d],
            ax=ax,
            fill=True,
            alpha=0.10,
            bw_adjust=bw_list[j],
            label=methods[j],
            color=colors[j],
            linewidth=2,
            cut=0
        )

    ax.axvline(
        x=true_val,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label="Truth" if plot_i == 0 else None,
        zorder=10
    )

    ax.set_xlim(x_low, x_high)

    ax.set_title(var_name_aug_latex[d], fontsize=24)
    ax.set_ylabel("")              # remove Density label
    ax.set_yticklabels([])         # remove y-axis numbers, keep ticks

    ax.tick_params(axis="x", labelsize=16)
    ax.tick_params(axis="y", length=4)

for k in range(len(plot_dims), len(axes)):
    axes[k].axis("off")


handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles, labels,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.01),
    ncol=5,
    fontsize=22,
    frameon=False
)

plt.subplots_adjust(
    left=0.035,
    right=0.995,
    bottom=0.22,
    top=0.95,
    wspace=0.1,
    hspace=0.5
)


plt.savefig("figs/SDEMEM_simu_kde.pdf", dpi=300, bbox_inches="tight")
plt.show()